In [4]:
import pandas as pd
import numpy as np
import os
from config_paths import USE_TEST_DATA, DATA_FOLDER
from config_variables import FILL_STRATEGIES, CATEGORICAL_VARS, CONTINUOUS_VARS, RECODE_MAPS

# CONFIGURATION
PRIMARY_WAVE = "o"
BACKUP_WAVES = [] if USE_TEST_DATA else ["n", "m", "l", "k"]
WAVE_PICKLE_DIR = f"../{DATA_FOLDER}/2_pickle_ukhls_waves"
OUTPUT_DIR  = f"../{DATA_FOLDER}/3_backfill_ukhls_waves"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"{PRIMARY_WAVE}_indresp_backfilled.pkl")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Targeting Wave {PRIMARY_WAVE} with backups from {BACKUP_WAVES}")

def safe_numeric(series):
    numeric = pd.to_numeric(series, errors='coerce')
    if pd.api.types.is_numeric_dtype(numeric):
        numeric = numeric.replace([np.inf, -np.inf], np.nan)
    return numeric

def get_base_code(col_name, wave_prefix):
    """Strip wave prefix to get the base variable code (e.g. 'o_age_dv' -> 'age_dv')."""
    prefix = f"{wave_prefix}_"
    return col_name[len(prefix):] if col_name.startswith(prefix) else col_name

def impute_column(series, strategy):
    """Apply a named fill strategy to a Series, returning the filled Series."""
    if strategy == "mode":
        fill_val = series.mode(dropna=True)
        return series.fillna(fill_val.iloc[0]) if not fill_val.empty else series
    elif strategy == "median":
        fill_val = series.median(skipna=True)
        return series.fillna(fill_val)
    elif strategy == "zero":
        return series.fillna(0)
    return series  # strategy is None — leave NaN

def snapshot_negatives(df):
    """Return a DataFrame of the same shape containing negative values where they
    exist and NaN elsewhere.  Used to restore 'most recent' negative responses
    for respondents where no backup wave has a positive answer."""
    snap = pd.DataFrame(np.nan, index=df.index, columns=df.columns)
    for col in df.columns:
        numeric = pd.to_numeric(df[col], errors='coerce')
        neg_mask = numeric < 0
        snap.loc[neg_mask, col] = df.loc[neg_mask, col]
    return snap

def mask_negatives(df):
    """Replace any negative numeric value with NaN so combine_first can patch them
    from older waves.  UKHLS uses negative codes (-1, -2, -8, -9, etc.) as
    'missing / inapplicable / refusal' sentinels — they are never real values."""
    masked = 0
    for col in df.columns:
        numeric = pd.to_numeric(df[col], errors='coerce')
        neg_mask = numeric < 0
        if neg_mask.any():
            masked += int(neg_mask.sum())
            df[col] = df[col].where(~neg_mask, other=np.nan)
    return df, masked

def build_expanded_master():
    primary_file = os.path.join(WAVE_PICKLE_DIR, f"{PRIMARY_WAVE}_indresp_optimized.pkl")

    if not os.path.exists(primary_file):
        raise FileNotFoundError(f"Could not find primary wave file: {primary_file}")

    # 1. Load Primary Wave
    print(f"Loading Primary Wave ({PRIMARY_WAVE})...")
    df_master = pd.read_pickle(primary_file).set_index('pidp')

    # Snapshot negative values BEFORE masking — used as fallback after backfill
    neg_snapshot = snapshot_negatives(df_master)

    # Mask negatives so backup waves can fill them
    df_master, n_masked = mask_negatives(df_master)
    if n_masked:
        print(f"   -> Masked {n_masked:,} negative sentinel values as NaN (ready for backfill).")

    # 2. Iteratively Backfill
    for i, wave in enumerate(BACKUP_WAVES):
        years_to_add = i + 1
        backup_file = os.path.join(WAVE_PICKLE_DIR, f"{wave}_indresp_optimized.pkl")

        if not os.path.exists(backup_file):
            print(f"Warning: {backup_file} not found. Skipping wave {wave}.")
            continue

        print(f"Processing Wave {wave} (Age offset: +{years_to_add})...")
        df_backup = pd.read_pickle(backup_file).set_index('pidp')

        # Align column names (e.g., n_age_dv -> o_age_dv)
        df_backup.columns = df_backup.columns.str.replace(f"{wave}_", f"{PRIMARY_WAVE}_")

        # Also mask negatives in backup so we don't backfill a bad value
        df_backup, _ = mask_negatives(df_backup)

        # A. Pull forward entire missing respondents
        new_ids = df_backup.index.difference(df_master.index)
        if not new_ids.empty:
            df_new = df_backup.loc[new_ids].copy()
            # Age increment logic
            age_col = f"{PRIMARY_WAVE}_age_dv"
            if age_col in df_new.columns:
                age_numeric = safe_numeric(df_new[age_col])
                df_new[age_col] = age_numeric + years_to_add

            df_master = pd.concat([df_master, df_new])
            # Extend snapshot for new respondents (all NaN — no primary-wave negatives)
            new_snap_rows = pd.DataFrame(np.nan, index=new_ids, columns=neg_snapshot.columns)
            neg_snapshot = pd.concat([neg_snapshot, new_snap_rows])
            print(f"   -> Added {len(new_ids):,} missing respondents.")

        # B. Patch holes in existing respondents (NaN + masked negatives)
        common_cols = df_master.columns.intersection(df_backup.columns)
        for col in common_cols:
            master_dtype = df_master[col].dtype
            backup_dtype = df_backup[col].dtype
            if isinstance(master_dtype, pd.CategoricalDtype) or isinstance(backup_dtype, pd.CategoricalDtype):
                df_master[col] = df_master[col].astype('object')
                df_backup[col] = df_backup[col].astype('object')

        df_master = df_master.combine_first(df_backup)
        print(f"   -> Patched missing variable values.")

    # 2.5 Restore most-recent negative where backfill found nothing better
    # Any cell that is still NaN but had a negative value in the primary wave
    # gets that negative value back, preserving "inapplicable" etc. over a
    # purely statistical imputation.
    print("\nRestoring primary-wave negatives where backfill found no positive answer...")
    restored = 0
    for col in df_master.columns:
        if col not in neg_snapshot.columns:
            continue
        still_null = df_master[col].isna()
        has_snap   = neg_snapshot[col].notna()
        mask = still_null & has_snap
        if mask.any():
            df_master.loc[mask, col] = neg_snapshot.loc[mask, col]
            restored += int(mask.sum())
    print(f"   -> Restored {restored:,} most-recent negative values.")

    # 3. Imputation — Apply per-variable fill strategies from config_variables.py
    # Only fires for cells that are STILL NaN (new respondents with no data in any wave)
    print("\nApplying fill strategies from config_variables.py...")
    df_master = df_master.reset_index()

    imputed = 0
    for col in df_master.columns:
        if 'idp' in col.lower():
            continue

        base = get_base_code(col, PRIMARY_WAVE)
        strategy = FILL_STRATEGIES.get(base)

        if strategy is not None:
            missing_before = df_master[col].isna().sum()
            df_master[col] = impute_column(df_master[col], strategy)
            filled = missing_before - df_master[col].isna().sum()
            if filled > 0:
                imputed += filled

    print(f"   -> Filled {imputed:,} missing values using per-variable strategies.")

    # 3.5 Value Recoding — Apply per-variable recode maps from config_variables.py
    print("Applying value recodes from config_variables.py...")
    recoded_cols = 0
    for col in df_master.columns:
        base = get_base_code(col, PRIMARY_WAVE)
        recode = RECODE_MAPS.get(base)
        if recode:
            df_master[col] = df_master[col].replace(recode)
            recoded_cols += 1
    print(f"   -> Recoded {recoded_cols} column(s).")

    # 4. Final Dtype Optimisation
    print("Optimizing dtypes...")
    for col in df_master.columns:
        if 'idp' in col.lower():
            id_numeric = safe_numeric(df_master[col]).fillna(0)
            df_master[col] = id_numeric.astype(np.int64)
            continue

        base = get_base_code(col, PRIMARY_WAVE)

        if base in CATEGORICAL_VARS:
            df_master[col] = df_master[col].astype('category')
        elif base in CONTINUOUS_VARS:
            df_master[col] = pd.to_numeric(safe_numeric(df_master[col]), downcast='float')
        elif pd.api.types.is_float_dtype(df_master[col]):
            df_master[col] = pd.to_numeric(safe_numeric(df_master[col]), downcast='float')
        elif pd.api.types.is_integer_dtype(df_master[col]):
            df_master[col] = safe_numeric(df_master[col]).astype('Int64')

    # 5. Save
    df_master.to_pickle(OUTPUT_FILE, protocol=5)
    print(f"DONE. Final Master size: {len(df_master):,} rows.")
    print(f"File saved to: {OUTPUT_FILE}")

build_expanded_master()


Targeting Wave o with backups from ['n', 'm', 'l', 'k']
Loading Primary Wave (o)...
   -> Masked 149,526 negative sentinel values as NaN (ready for backfill).
Processing Wave n (Age offset: +1)...
   -> Added 6,503 missing respondents.
   -> Patched missing variable values.
Processing Wave m (Age offset: +2)...
   -> Added 2,450 missing respondents.
   -> Patched missing variable values.
Processing Wave l (Age offset: +3)...
   -> Added 2,460 missing respondents.
   -> Patched missing variable values.
Processing Wave k (Age offset: +4)...
   -> Added 3,092 missing respondents.
   -> Patched missing variable values.

Restoring primary-wave negatives where backfill found no positive answer...
   -> Restored 119,099 most-recent negative values.

Applying fill strategies from config_variables.py...
   -> Filled 142,331 missing values using per-variable strategies.
Applying value recodes from config_variables.py...
   -> Recoded 4 column(s).
Optimizing dtypes...
DONE. Final Master size: 47,